# 02 - Classificação: Regressão Logística, LDA, QDA e KNN

## Por que usar modelos de classificação?

Usamos algoritmos de **Classificação** quando a variável resposta ($Y$) é **qualitativa/categórica** (ex.: Inadimplente vs. Não Inadimplente, Fraude vs. Legítimo, Diagnóstico Positivo vs. Negativo).

Se tentarmos ajustar uma **Regressão Linear OLS** para prever uma probabilidade binária ($Y \in \{0, 1\}$):
1. As previsões estimadas $\hat{Y}$ podem extrapolar o intervalo válido de probabilidades, resultando em valores $< 0$ ou $> 1$.
2. A variância dos resíduos não é constante (violação severa de homocedasticidade).

---

## Fundamentação: A Regressão Logística

Para garantir que a saída represente sempre uma probabilidade válida $p(X) = P(Y=1 \mid X) \in [0, 1]$, a regressão logística aplica a **função sigmoide/logística** sobre uma combinação linear dos preditores:

$$p(X) = \frac{e^{\beta_0 + \beta_1 X_1 + \dots + \beta_p X_p}}{1 + e^{\beta_0 + \beta_1 X_1 + \dots + \beta_p X_p}}$$

Manipulando algebricamente, obtemos a **Razão de Chances (*Odds*)**:

$$\frac{p(X)}{1 - p(X)} = e^{\beta_0 + \beta_1 X_1 + \dots + \beta_p X_p}$$

Tomando o logaritmo natural em ambos os lados, chegamos ao **Logit** ou **Log-Odds**:

$$\log\left(\frac{p(X)}{1 - p(X)}\right) = \beta_0 + \beta_1 X_1 + \dots + \beta_p X_p$$

* **Interpretação de $\beta_j$:** Um aumento de 1 unidade em $X_j$ altera o *log-odds* em $\beta_j$, ou multiplica as chances (*odds*) por um fator de $e^{\beta_j}$, mantendo os demais preditores constantes.

---

## Como o modelo é treinado? (Máxima Verossimilhança)

Diferente da regressão linear (que minimiza o RSS por mínimos quadrados), a regressão logística estima $\boldsymbol{\beta}$ maximizando a **Função de Verossimilhança (*Likelihood*)**:

$$\ell(\boldsymbol{\beta}) = \prod_{i: y_i = 1} p(x_i) \prod_{i': y_{i'} = 0} (1 - p(x_{i'}))$$

Computacionalmente, minimiza-se a perda logarítmica (*Log-Loss* / *Binary Cross-Entropy*), geralmente via métodos iterativos de otimização como Newton-Raphson ou gradiente descendente.

In [1]:
# Importa bibliotecas numéricas e tabulares
import numpy as np
import pandas as pd

# Importa bibliotecas de visualização
import matplotlib.pyplot as plt
import seaborn as sns

# Ferramentas de modelagem e validação do scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Define tema visual padronizado
sns.set_theme(style="whitegrid", context="notebook")

## Carregando os Dados: Dataset Default (ISLP)

Utilizamos o clássico dataset **`Default`** do ISLP (ou simulamos sua estrutura com dados sintéticos idênticos caso a biblioteca `ISLP` não esteja instalada no ambiente). 

* **Objetivo:** Prever se um cliente será inadimplente no cartão de crédito (`default`: Yes/No).
* **Preditores:**
  * `student`: se o cliente é estudante (Yes/No)
  * `balance`: saldo médio devedor remanescente no cartão após o pagamento mensal
  * `income`: renda anual do cliente

In [3]:
# Tenta carregar do pacote ISLP; se não estiver instalado, carrega via URL do repositório oficial
try:
    from ISLP import load_data
    Default = load_data('Default')
except ImportError:
    url = "https://raw.githubusercontent.com/intro-stat-learning/ISLP_labs/v2/ISLP/data/Default.csv"
    Default = pd.read_csv(url)
    if 'Unnamed: 0' in Default.columns:
        Default = Default.drop(columns=['Unnamed: 0'])

# Converte variáveis qualitativas binárias para numéricas (0 e 1)
Default['default_num'] = (Default['default'] == 'Yes').astype(int)
Default['student_num'] = (Default['student'] == 'Yes').astype(int)

print("Dimensões do dataset:", Default.shape)
print("Distribuição do alvo (Default):\n", Default['default'].value_counts(normalize=True))
Default.head()

HTTPError: HTTP Error 404: Not Found

## Implementação: Regressão Logística Simples

Iniciamos estimando a probabilidade de inadimplência utilizando exclusivamente o saldo do cartão (`balance`):

$$P(\text{default} = \text{Yes} \mid \text{balance}) = \frac{e^{\beta_0 + \beta_1 \cdot \text{balance}}}{1 + e^{\beta_0 + \beta_1 \cdot \text{balance}}}$$

In [ ]:
# Seleciona preditor e alvo
X_simple = Default[['balance']]
y = Default['default_num']

# Divisão em treino e teste (75% / 25%) estratificada devido ao desbalanceamento
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_simple,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# Instancia e treina o estimador logístico
# C=1e5 para desativar penalização de regularização L2 e aproximar da solução clássica OLS/MLE
simple_logit = LogisticRegression(C=1e5, solver='lbfgs')
simple_logit.fit(X_train_s, y_train_s)

beta_0 = simple_logit.intercept_[0]
beta_1 = simple_logit.coef_[0][0]

print(f"Intercepto (beta_0): {beta_0:.4f}")
print(f"Coeficiente para balance (beta_1): {beta_1:.6f}")
print(f"Razão de chances (Odds Ratio por 100 dólares de saldo): {np.exp(beta_1 * 100):.4f}")

In [ ]:
# Gera grade contínua para curva preditiva
balance_grid = np.linspace(Default['balance'].min(), Default['balance'].max(), 300).reshape(-1, 1)
prob_pred = simple_logit.predict_proba(balance_grid)[:, 1]

# Visualização da curva sigmoide vs dados reais
plt.figure(figsize=(10, 5))
plt.scatter(Default['balance'], Default['default_num'], alpha=0.2, color='gray', label='Observado (0 ou 1)')
plt.plot(balance_grid, prob_pred, color='crimson', linewidth=3, label='P(Default = Yes | Balance)')
plt.axhline(0.5, color='black', linestyle='dotted', alpha=0.7, label='Limiar padrão (0.5)')
plt.xlabel('Saldo Médio Remanescente ($)')
plt.ylabel('Probabilidade Estimada de Inadimplência')
plt.title('Regressão Logística Simples: Curva Sigmoide')
plt.legend()
plt.show()

## Avaliação: Matriz de Confusão e o Efeito do Limiar (*Threshold*)

Em problemas com desbalanceamento severo de classes (~3.3% de inadimplentes), a métrica de **Acurácia** é enganosa (um modelo que chuta sempre "Não" teria 96.7% de acurácia, mas utilidade nula para risco). 

Analisamos:
* **Sensibilidade / Recall (Taxa de Verdadeiro Positivo):** fração de inadimplentes reais que o modelo conseguiu capturar.
* **Especificidade:** fração de adimplentes corretos identificados.

In [ ]:
# Previsões de probabilidade no teste
y_prob_s = simple_logit.predict_proba(X_test_s)[:, 1]

# Classificação com limiar padrão de 0.5
y_pred_50 = (y_prob_s >= 0.5).astype(int)

# Classificação com limiar de negócio de 0.2 (priorizando detecção de risco)
y_pred_20 = (y_prob_s >= 0.2).astype(int)

cm_50 = confusion_matrix(y_test_s, y_pred_50)
cm_20 = confusion_matrix(y_test_s, y_pred_20)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.heatmap(cm_50, annot=True, fmt='d', cmap='Blues', ax=axes[0], cbar=False)
axes[0].set_title('Limiar = 0.50 (Padrão)\nRecall: {:.2%}'.format(recall_score(y_test_s, y_pred_50)))
axes[0].set_xlabel('Predito')
axes[0].set_ylabel('Real')

sns.heatmap(cm_20, annot=True, fmt='d', cmap='Oranges', ax=axes[1], cbar=False)
axes[1].set_title('Limiar = 0.20 (Conservador)\nRecall: {:.2%}'.format(recall_score(y_test_s, y_pred_20)))
axes[1].set_xlabel('Predito')
axes[1].set_ylabel('Real')

plt.tight_layout()
plt.show()

## Checkpoint para o Engenheiro: A Curva ROC e o Trade-Off de Decisão

Em sistemas de Machine Learning de produção, o modelo raramente cospe classes binárias rígidas; ele cospe **probabilidades calibradas**. A escolha do limiar de corte ($threshold$) é uma **decisão financeira/operacional** baseada na matriz de custos:

* **Custo de Falso Negativo (Liberar crédito para um caloteiro):** Perda total do saldo emprestado (custo alto).
* **Custo de Falso Positivo (Bloquear crédito de um bom pagador):** Atrito e perda da taxa de transação (custo menor).

A métrica **ROC-AUC** mede a capacidade do modelo de ordenar clientes de maior risco acima dos de menor risco, independente do limiar escolhido.

## Exercício Prático: Regressão Múltipla e Comparativo com LDA, QDA e KNN

Agora treinamos o modelo de **Regressão Logística Múltipla** com todos os preditores (`balance`, `income`, `student_num`) e comparamos seu desempenho contra outros classificadores canônicos do ISLP:
1. **LDA (*Linear Discriminant Analysis*):** Assume densidades gaussianas com matriz de covariância compartilhada.
2. **QDA (*Quadratic Discriminant Analysis*):** Assume densidades gaussianas com covariância própria por classe (fronteiras quadráticas).
3. **KNN (*K-Nearest Neighbors*):** Classificador não paramétrico baseado em distância local ($K=10$).

In [ ]:
# 1. Preparação da matriz multivariada de features
features = ['balance', 'income', 'student_num']
X_multi = Default[features]
y_multi = Default['default_num']

X_tr, X_te, y_tr, y_te = train_test_split(
    X_multi,
    y_multi,
    test_size=0.25,
    random_state=42,
    stratify=y_multi
)

# 2. Treinamento dos modelos
models = {
    'Regressão Logística': LogisticRegression(C=1e5, solver='lbfgs'),
    'LDA': LinearDiscriminantAnalysis(),
    'QDA': QuadraticDiscriminantAnalysis(),
    'KNN (K=10)': KNeighborsClassifier(n_neighbors=10)
}

results_dict = {}

plt.figure(figsize=(9, 6))

for name, clf in models.items():
    clf.fit(X_tr, y_tr)
    y_pred = clf.predict(X_te)
    y_prob = clf.predict_proba(X_te)[:, 1]
    
    auc = roc_auc_score(y_te, y_prob)
    rec = recall_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred, zero_division=0)
    
    results_dict[name] = {
        'ROC-AUC': auc,
        'Recall (0.5)': rec,
        'Precisão (0.5)': prec,
        'Acurácia': accuracy_score(y_te, y_pred)
    }
    
    fpr, tpr, _ = roc_curve(y_te, y_prob)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})")

plt.plot([0, 1], [0, 1], 'k--', label='Aleatório (AUC = 0.500)')
plt.xlabel('Taxa de Falso Positivo (1 - Especificidade)')
plt.ylabel('Taxa de Verdadeiro Positivo (Recall)')
plt.title('Curvas ROC - Comparativo de Modelos (Capítulo 4 ISLP)')
plt.legend()
plt.show()

# Exibe métricas consolidadas
pd.DataFrame(results_dict).T

In [ ]:
# Analisando os coeficientes da Regressão Logística Múltipla
logit_multi = models['Regressão Logística']

coef_multi_df = pd.DataFrame({
    'Feature': features,
    'Coeficiente (Beta)': logit_multi.coef_[0],
    'Odds Ratio [exp(Beta)]': np.exp(logit_multi.coef_[0])
}).sort_values(by='Coeficiente (Beta)', ascending=False)

print("Intercepto:", logit_multi.intercept_[0])
print("\nImpacto dos Coeficientes:")
coef_multi_df

## Diagnóstico Crítico: O Paradoxo do Estudante

Ao rodar uma regressão logística univariada com `student`, o coeficiente é **positivo** (estudantes inadimplam mais no agregado).

Porém, na **Regressão Múltipla** (controlando por `balance`), o coeficiente de `student_num` torna-se **negativo**! 

* **Por que isso ocorre?** 
  Estudantes tendem a carregar saldos devedores (`balance`) muito maiores do que não estudantes. Para um mesmo nível de saldo devedor fixo, um estudante é, na verdade, **menos arriscado** do que um não estudante. Este é um caso clássico do **Paradoxo de Simpson** e reforça a importância da modelagem multivariada.

### Report Executivo (Decisão de Negócio)

* **Objetivo do Modelo:** Estimar a probabilidade de inadimplência de crédito para política de concessão e alertas antecipados de risco.
* **Resultado Principal:** O saldo devedor mensal (`balance`) é o driver dominante de inadimplência. Estudantes apresentam menor risco intrínseco quando equalizado o saldo devedor.
* **Qualidade Preditiva:** Desempenho de ranqueamento de excelência (ROC-AUC $\approx 0.95$). Os modelos paramétricos (Regressão Logística e LDA) superaram o KNN devido à dimensionalidade e estrutura linear do problema.
* **Calibração de Negócio:** O limiar padrão de 0.50 deixa passar calotes críticos. Recomenda-se calibrar a esteira para um limiar de corte de 0.20 a 0.25 para maximizar a captura de clientes de risco (elevando o Recall de ~30% para >70%).
* **Recomendação de Uso:** Aprovado para implantação em esteira de risco para atribuição automática de limites e ações de cobrança preventiva.